# Silver -> Blob CSV (bulk-load staging for Azure SQL OLTP)

Fast alternative to the row-by-row JDBC load in `50-silver-to-azuresql-oltp.ipynb`.

## Data flow
```
retail_lakehouse.silver.*  (Delta)
   -> Azure Blob Storage   <container>/<prefix>/<table>.csv   (this notebook)
   -> Azure SQL retail.*    via BULK INSERT (staging temp table)  (bulk-load SQL)
```

## Why
Azure SQL DB cannot BULK-read OneLake directly, so the silver tables are reshaped
with the **exact same transforms as notebook 50** (imported verbatim below) and
written as RFC4180 CSV to blob. Then `deploy/azuresql/bulk-load/10-bulk-load.sql`
loads each file with `BULK INSERT` into a staging temp table, then INSERT...SELECT,
which is dramatically faster and more resilient than per-row JDBC inserts.

## Column contract
Each CSV holds every OLTP column **except the IDENTITY surrogate PK** (the server
assigns it on INSERT), in DDL order, `loaded_at` last. The CSV column order matches
the staging `CREATE TABLE (...)` and the `INSERT` column list in the SQL script
exactly -- both are generated from the same DDL. `BIT` columns are emitted as 0/1.

## Prerequisites
- OLTP schema deployed (`deploy/azuresql/deploy-schema.ps1`).
- A blob container reachable from Fabric; a SAS token (or account key) with write
  access for this notebook and (for the SQL loader) a read/list SAS.

## Note on scale
`SINGLE_FILE=True` (default) writes one `<table>.csv` per table -- simplest to load
with the committed static SQL, ideal for testing. For very large tables set
`SINGLE_FILE=False` to write parallel part files; the RUN cell then prints a
generated per-file loader (Azure SQL DB `BULK INSERT` reads one file per statement).


In [ ]:
from pyspark.sql import functions as F
from datetime import datetime, timezone
import os

try:
    import notebookutils  # Fabric
except ImportError:  # pragma: no cover - older runtimes
    import mssparkutils as notebookutils

In [ ]:
# =============================================================================
# PARAMETERS  (Fabric "parameters" cell)
# =============================================================================

import os


def get_env(var_name, default=None):
    return os.environ.get(var_name, default)


# --- source lakehouse ---
LAKEHOUSE_NAME = get_env("LAKEHOUSE_NAME", "retail_lakehouse")
SILVER_DB      = get_env("SILVER_DB", "silver")

# --- target blob storage ---
BLOB_ACCOUNT   = get_env("BLOB_ACCOUNT", "")      # storage account name (no suffix)
BLOB_CONTAINER = get_env("BLOB_CONTAINER", "")    # container name
BLOB_PREFIX    = get_env("BLOB_PREFIX", "oltp-export")  # folder prefix within container
# One of these must be set to authenticate the write:
BLOB_SAS       = get_env("BLOB_SAS", "")          # SAS token (with or without leading '?')
BLOB_ACCOUNT_KEY = get_env("BLOB_ACCOUNT_KEY", "")  # storage account key (alternative)

# --- export behaviour ---
SINGLE_FILE       = str(get_env("SINGLE_FILE", "true")).lower() == "true"
EXPORT_PARTITIONS = int(get_env("EXPORT_PARTITIONS", "0"))  # 0 = Spark default (multi-file mode)
ONLY_TABLES       = get_env("ONLY_TABLES", "")   # comma-separated; parsed next cell
LOADED_AT         = get_env("LOADED_AT", "")     # blank -> run time (UTC)


In [ ]:
# =============================================================================
# DERIVED CONFIG  (runs after any injected parameters)
# =============================================================================
from datetime import datetime, timezone

assert BLOB_ACCOUNT and BLOB_CONTAINER, "Set BLOB_ACCOUNT and BLOB_CONTAINER"

# comma-separated string -> list (default = all tables)
if isinstance(ONLY_TABLES, str):
    ONLY_TABLES = [t.strip() for t in ONLY_TABLES.split(",") if t.strip()]

# one UTC batch timestamp per run, stamped on every row (unless supplied)
if not LOADED_AT:
    LOADED_AT = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]

# deterministic UTC timestamp rendering in the CSV
spark.sql("SET spark.sql.session.timeZone = UTC")  # noqa: F821

# Write via the ABFS driver (dfs endpoint). AUTH_METHOD controls credentials:
#   "aad" (default) - Fabric passes the running user's AAD identity through to
#         the external account (the user needs Storage Blob Data RBAC). No SAS or
#         key needed, and it avoids the FixedSASTokenProvider class that the
#         Fabric Spark runtime does not ship (so ABFS+SAS fails to initialize).
#   "key" - shared-account-key (SharedKey); needs BLOB_ACCOUNT_KEY and the
#         account must allow shared-key access.
#   "sas" - fixed SAS token (only if the runtime provides FixedSASTokenProvider).
# The Azure SQL loader reads the same files via the blob endpoint using its own
# SAS credential (a container SAS is valid on both blob and dfs endpoints).
AUTH_METHOD = str(get_env("AUTH_METHOD", "aad")).lower()
_host = f"{BLOB_ACCOUNT}.dfs.core.windows.net"
if AUTH_METHOD == "key":
    assert BLOB_ACCOUNT_KEY, "AUTH_METHOD=key needs BLOB_ACCOUNT_KEY"
    spark.conf.set(f"fs.azure.account.auth.type.{_host}", "SharedKey")  # noqa: F821
    spark.conf.set(f"fs.azure.account.key.{_host}", BLOB_ACCOUNT_KEY)  # noqa: F821
elif AUTH_METHOD == "sas":
    assert BLOB_SAS, "AUTH_METHOD=sas needs BLOB_SAS"
    _sas = BLOB_SAS[1:] if BLOB_SAS.startswith("?") else BLOB_SAS
    spark.conf.set(f"fs.azure.account.auth.type.{_host}", "SAS")  # noqa: F821
    spark.conf.set(  # noqa: F821
        f"fs.azure.sas.token.provider.type.{_host}",
        "org.apache.hadoop.fs.azurebfs.sas.FixedSASTokenProvider",
    )
    spark.conf.set(f"fs.azure.sas.fixed.token.{_host}", _sas)  # noqa: F821
# AUTH_METHOD == "aad": rely on Fabric AAD passthrough; set no auth conf.

BLOB_BASE = f"abfss://{BLOB_CONTAINER}@{_host}/{BLOB_PREFIX}"

# authoritative OLTP columns (IDENTITY PK excluded), DDL order, loaded_at last.
# Generated from deploy/azuresql/schema/*.sql -- keep in sync with the bulk-load SQL.
TABLE_DDL = {
    "geographies": [("geography_id", "BIGINT"), ("city", "NVARCHAR(100)"), ("state", "NVARCHAR(50)"), ("zip_code", "VARCHAR(15)"), ("district", "NVARCHAR(100)"), ("region", "NVARCHAR(100)"), ("loaded_at", "DATETIME2(3)")],
    "customers": [("customer_id", "BIGINT"), ("first_name", "NVARCHAR(100)"), ("last_name", "NVARCHAR(100)"), ("address", "NVARCHAR(255)"), ("geography_id", "BIGINT"), ("loyalty_card", "VARCHAR(50)"), ("phone", "VARCHAR(30)"), ("ble_id", "VARCHAR(64)"), ("ad_id", "VARCHAR(64)"), ("loaded_at", "DATETIME2(3)")],
    "stores": [("store_id", "BIGINT"), ("store_number", "VARCHAR(20)"), ("address", "NVARCHAR(255)"), ("geography_id", "BIGINT"), ("tax_rate", "DECIMAL(6,4)"), ("volume_class", "VARCHAR(20)"), ("store_format", "VARCHAR(30)"), ("operating_hours", "VARCHAR(50)"), ("daily_traffic_multiplier", "DECIMAL(9,4)"), ("loaded_at", "DATETIME2(3)")],
    "distribution_centers": [("dc_id", "BIGINT"), ("dc_number", "VARCHAR(20)"), ("address", "NVARCHAR(255)"), ("geography_id", "BIGINT"), ("loaded_at", "DATETIME2(3)")],
    "trucks": [("truck_id", "BIGINT"), ("license_plate", "VARCHAR(20)"), ("refrigeration", "BIT"), ("dc_id", "BIGINT"), ("loaded_at", "DATETIME2(3)")],
    "products": [("product_id", "BIGINT"), ("product_name", "NVARCHAR(200)"), ("brand", "NVARCHAR(100)"), ("company", "NVARCHAR(100)"), ("department", "NVARCHAR(100)"), ("category", "NVARCHAR(100)"), ("subcategory", "NVARCHAR(100)"), ("cost", "DECIMAL(19,4)"), ("msrp", "DECIMAL(19,4)"), ("sale_price", "DECIMAL(19,4)"), ("requires_refrigeration", "BIT"), ("launch_date", "DATETIME2(3)"), ("taxability", "VARCHAR(30)"), ("tags", "NVARCHAR(500)"), ("loaded_at", "DATETIME2(3)")],
    "sales": [("receipt_id", "VARCHAR(64)"), ("store_id", "BIGINT"), ("customer_id", "BIGINT"), ("sale_ts", "DATETIME2(3)"), ("receipt_type", "VARCHAR(30)"), ("tender_type", "VARCHAR(30)"), ("payment_method", "VARCHAR(30)"), ("subtotal_amount", "DECIMAL(19,4)"), ("discount_amount", "DECIMAL(19,4)"), ("tax_amount", "DECIMAL(19,4)"), ("total_amount", "DECIMAL(19,4)"), ("promo_code", "VARCHAR(40)"), ("trace_id", "VARCHAR(64)"), ("loaded_at", "DATETIME2(3)")],
    "sale_lines": [("receipt_id", "VARCHAR(64)"), ("line_number", "INT"), ("product_id", "BIGINT"), ("quantity", "INT"), ("unit_price", "DECIMAL(19,4)"), ("extended_price", "DECIMAL(19,4)"), ("promo_code", "VARCHAR(40)"), ("discount_amount", "DECIMAL(19,4)"), ("loaded_at", "DATETIME2(3)")],
    "online_orders": [("order_id", "VARCHAR(64)"), ("customer_id", "BIGINT"), ("order_ts", "DATETIME2(3)"), ("subtotal_amount", "DECIMAL(19,4)"), ("tax_amount", "DECIMAL(19,4)"), ("total_amount", "DECIMAL(19,4)"), ("payment_method", "VARCHAR(30)"), ("loaded_at", "DATETIME2(3)")],
    "online_order_lines": [("order_id", "VARCHAR(64)"), ("line_number", "INT"), ("product_id", "BIGINT"), ("quantity", "INT"), ("unit_price", "DECIMAL(19,4)"), ("extended_price", "DECIMAL(19,4)"), ("promo_code", "VARCHAR(40)"), ("fulfillment_mode", "VARCHAR(30)"), ("fulfillment_status", "VARCHAR(30)"), ("node_type", "VARCHAR(30)"), ("node_id", "BIGINT"), ("picked_ts", "DATETIME2(3)"), ("shipped_ts", "DATETIME2(3)"), ("delivered_ts", "DATETIME2(3)"), ("loaded_at", "DATETIME2(3)")],
    "payments": [("receipt_id", "VARCHAR(64)"), ("order_id", "VARCHAR(64)"), ("payment_ts", "DATETIME2(3)"), ("payment_method", "VARCHAR(30)"), ("amount", "DECIMAL(19,4)"), ("transaction_id", "VARCHAR(64)"), ("status", "VARCHAR(20)"), ("decline_reason", "VARCHAR(100)"), ("processing_time_ms", "BIGINT"), ("store_id", "BIGINT"), ("customer_id", "BIGINT"), ("loaded_at", "DATETIME2(3)")],
    "inventory_transactions": [("location_type", "VARCHAR(10)"), ("store_id", "BIGINT"), ("dc_id", "BIGINT"), ("product_id", "BIGINT"), ("txn_ts", "DATETIME2(3)"), ("txn_type", "VARCHAR(30)"), ("quantity", "INT"), ("balance", "INT"), ("source", "VARCHAR(50)"), ("trace_id", "VARCHAR(64)"), ("loaded_at", "DATETIME2(3)")],
    "reorders": [("reorder_ts", "DATETIME2(3)"), ("store_id", "BIGINT"), ("dc_id", "BIGINT"), ("product_id", "BIGINT"), ("current_quantity", "INT"), ("reorder_quantity", "INT"), ("reorder_point", "INT"), ("priority", "VARCHAR(20)"), ("trace_id", "VARCHAR(64)"), ("loaded_at", "DATETIME2(3)")],
    "stockouts": [("stockout_ts", "DATETIME2(3)"), ("store_id", "BIGINT"), ("dc_id", "BIGINT"), ("product_id", "BIGINT"), ("last_known_quantity", "INT"), ("trace_id", "VARCHAR(64)"), ("loaded_at", "DATETIME2(3)")],
    "shipment_movements": [("shipment_number", "VARCHAR(64)"), ("truck_id", "BIGINT"), ("dc_id", "BIGINT"), ("store_id", "BIGINT"), ("status", "VARCHAR(30)"), ("event_ts", "DATETIME2(3)"), ("eta", "DATETIME2(3)"), ("etd", "DATETIME2(3)"), ("departure_time", "DATETIME2(3)"), ("actual_unload_duration", "DECIMAL(9,2)"), ("trace_id", "VARCHAR(64)"), ("loaded_at", "DATETIME2(3)")],
    "shipment_lines": [("shipment_number", "VARCHAR(64)"), ("truck_id", "BIGINT"), ("product_id", "BIGINT"), ("quantity", "INT"), ("action", "VARCHAR(20)"), ("location_id", "BIGINT"), ("location_type", "VARCHAR(10)"), ("event_ts", "DATETIME2(3)"), ("trace_id", "VARCHAR(64)"), ("loaded_at", "DATETIME2(3)")],
}

TABLE_COLUMNS = {t: [c for c, _ in cols] for t, cols in TABLE_DDL.items()}

print(f"source={LAKEHOUSE_NAME}.{SILVER_DB} -> {BLOB_BASE}")
print(f"single_file={SINGLE_FILE}  export_partitions={EXPORT_PARTITIONS}  loaded_at={LOADED_AT}")


In [ ]:
# =============================================================================
# TRANSFORM HELPERS
# =============================================================================

def silver(table):
    return spark.table(f"{LAKEHOUSE_NAME}.{SILVER_DB}.{table}")  # noqa: F821


def money_cents(col):
    """Integer cents -> DECIMAL(19,4) dollars."""
    return (F.col(col) / F.lit(100.0)).cast("decimal(19,4)")

In [ ]:
# =============================================================================
# MASTER / REFERENCE TRANSFORMS
# =============================================================================

def build_geographies():
    return silver("dim_geographies").select(
        F.col("ID").alias("geography_id"),
        F.col("City").alias("city"),
        F.col("State").alias("state"),
        F.col("ZipCode").alias("zip_code"),
        F.col("District").alias("district"),
        F.col("Region").alias("region"),
    )


def build_customers():
    return silver("dim_customers").select(
        F.col("ID").alias("customer_id"),
        F.col("FirstName").alias("first_name"),
        F.col("LastName").alias("last_name"),
        F.col("Address").alias("address"),
        F.col("GeographyID").alias("geography_id"),
        F.col("LoyaltyCard").alias("loyalty_card"),
        F.col("Phone").alias("phone"),
        F.col("BLEId").alias("ble_id"),
        F.col("AdId").alias("ad_id"),
    )


def build_stores():
    return silver("dim_stores").select(
        F.col("ID").alias("store_id"),
        F.col("StoreNumber").alias("store_number"),
        F.col("Address").alias("address"),
        F.col("GeographyID").alias("geography_id"),
        F.col("tax_rate").cast("decimal(6,4)").alias("tax_rate"),
        F.col("volume_class"),
        F.col("store_format"),
        F.col("operating_hours"),
        F.col("daily_traffic_multiplier").cast("decimal(9,4)").alias("daily_traffic_multiplier"),
    )


def build_distribution_centers():
    return silver("dim_distribution_centers").select(
        F.col("ID").alias("dc_id"),
        F.col("DCNumber").alias("dc_number"),
        F.col("Address").alias("address"),
        F.col("GeographyID").alias("geography_id"),
    )


def build_trucks():
    return silver("dim_trucks").select(
        F.col("ID").alias("truck_id"),
        F.col("LicensePlate").alias("license_plate"),
        F.col("Refrigeration").alias("refrigeration"),
        F.col("DCID").cast("long").alias("dc_id"),
    )


def build_products():
    return silver("dim_products").select(
        F.col("ID").alias("product_id"),
        F.col("ProductName").alias("product_name"),
        F.col("Brand").alias("brand"),
        F.col("Company").alias("company"),
        F.col("Department").alias("department"),
        F.col("Category").alias("category"),
        F.col("Subcategory").alias("subcategory"),
        F.col("Cost").cast("decimal(19,4)").alias("cost"),
        F.col("MSRP").cast("decimal(19,4)").alias("msrp"),
        F.col("SalePrice").cast("decimal(19,4)").alias("sale_price"),
        F.col("RequiresRefrigeration").alias("requires_refrigeration"),
        F.col("LaunchDate").alias("launch_date"),
        F.col("taxability"),
        F.col("Tags").alias("tags"),
    )

In [ ]:
# =============================================================================
# COMMERCE TRANSFORMS
# =============================================================================

def build_sales():
    receipts = silver("fact_receipts")
    # receipt-level promo folded onto the header (one code per receipt)
    promo = (
        silver("fact_promotions")
        .groupBy("receipt_id_ext")
        .agg(F.first("promo_code", ignorenulls=True).alias("promo_code"))
    )
    return (
        receipts.join(promo, "receipt_id_ext", "left").select(
            F.col("receipt_id_ext").alias("receipt_id"),
            F.col("store_id"),
            F.col("customer_id"),
            F.col("event_ts").alias("sale_ts"),
            F.col("receipt_type"),
            F.col("tender_type"),
            F.col("payment_method"),
            money_cents("subtotal_cents").alias("subtotal_amount"),
            F.col("discount_amount").cast("decimal(19,4)").alias("discount_amount"),
            money_cents("tax_cents").alias("tax_amount"),
            money_cents("total_cents").alias("total_amount"),
            F.col("promo_code"),
            F.col("trace_id"),
        )
    )


def build_sale_lines():
    lines = silver("fact_receipt_lines")
    # line-level promo discount folded onto the line
    promo = (
        silver("fact_promo_lines")
        .select(
            F.col("receipt_id_ext").alias("pl_receipt"),
            F.col("line_number").alias("pl_line"),
            (F.col("discount_cents") / F.lit(100.0)).cast("decimal(19,4)").alias("discount_amount"),
        )
        .groupBy("pl_receipt", "pl_line")
        .agg(F.sum("discount_amount").alias("discount_amount"))
    )
    join_cond = (lines["receipt_id_ext"] == promo["pl_receipt"]) & (lines["line_num"] == promo["pl_line"])
    return (
        lines.join(promo, join_cond, "left").select(
            lines["receipt_id_ext"].alias("receipt_id"),
            lines["line_num"].alias("line_number"),
            lines["product_id"],
            lines["quantity"].cast("int").alias("quantity"),
            money_cents("unit_cents").alias("unit_price"),
            money_cents("ext_cents").alias("extended_price"),
            lines["promo_code"],
            promo["discount_amount"],
        )
    )


def build_online_orders():
    return silver("fact_online_order_headers").select(
        F.col("order_id_ext").alias("order_id"),
        F.col("customer_id"),
        F.col("event_ts").alias("order_ts"),
        money_cents("subtotal_cents").alias("subtotal_amount"),
        money_cents("tax_cents").alias("tax_amount"),
        money_cents("total_cents").alias("total_amount"),
        F.col("payment_method"),
    )


def build_online_order_lines():
    return silver("fact_online_order_lines").select(
        F.col("order_id"),
        F.col("line_num").alias("line_number"),
        F.col("product_id"),
        F.col("quantity").cast("int").alias("quantity"),
        money_cents("unit_cents").alias("unit_price"),
        money_cents("ext_cents").alias("extended_price"),
        F.col("promo_code"),
        F.col("fulfillment_mode"),
        F.col("fulfillment_status"),
        F.col("node_type"),
        F.col("node_id"),
        F.col("picked_ts"),
        F.col("shipped_ts"),
        F.col("delivered_ts"),
    )


def build_payments():
    return silver("fact_payments").select(
        F.col("receipt_id_ext").alias("receipt_id"),
        F.col("order_id_ext").alias("order_id"),
        F.col("event_ts").alias("payment_ts"),
        F.col("payment_method"),
        money_cents("amount_cents").alias("amount"),
        F.col("transaction_id"),
        F.col("status"),
        F.col("decline_reason"),
        F.col("processing_time_ms"),
        F.col("store_id"),
        F.col("customer_id"),
    )

In [ ]:
# =============================================================================
# SUPPLY-CHAIN TRANSFORMS
# =============================================================================

def build_inventory_transactions():
    store = silver("fact_store_inventory_txn").select(
        F.lit("STORE").alias("location_type"),
        F.col("store_id").cast("long").alias("store_id"),
        F.lit(None).cast("long").alias("dc_id"),
        F.col("product_id"),
        F.col("event_ts").alias("txn_ts"),
        F.col("txn_type"),
        F.col("quantity").cast("int").alias("quantity"),
        F.col("balance").cast("int").alias("balance"),
        F.col("source"),
        F.col("trace_id"),
    )
    dc = silver("fact_dc_inventory_txn").select(
        F.lit("DC").alias("location_type"),
        F.lit(None).cast("long").alias("store_id"),
        F.col("dc_id").cast("long").alias("dc_id"),
        F.col("product_id"),
        F.col("event_ts").alias("txn_ts"),
        F.col("txn_type"),
        F.col("quantity").cast("int").alias("quantity"),
        F.col("balance").cast("int").alias("balance"),
        F.col("Source").alias("source"),
        F.col("trace_id"),
    )
    return store.unionByName(dc)


def build_reorders():
    return silver("fact_reorders").select(
        F.col("event_ts").alias("reorder_ts"),
        F.col("store_id").cast("long").alias("store_id"),
        F.col("dc_id").cast("long").alias("dc_id"),
        F.col("product_id"),
        F.col("current_quantity").cast("int").alias("current_quantity"),
        F.col("reorder_quantity").cast("int").alias("reorder_quantity"),
        F.col("reorder_point").cast("int").alias("reorder_point"),
        F.col("priority"),
        F.col("trace_id"),
    )


def build_stockouts():
    return silver("fact_stockouts").select(
        F.col("event_ts").alias("stockout_ts"),
        F.col("StoreID").cast("long").alias("store_id"),
        F.col("DCID").cast("long").alias("dc_id"),
        F.col("ProductID").cast("long").alias("product_id"),
        F.col("LastKnownQuantity").cast("int").alias("last_known_quantity"),
        F.col("trace_id"),
    )


def build_shipment_movements():
    return silver("fact_truck_moves").select(
        F.col("shipment_id").alias("shipment_number"),
        F.col("truck_id").cast("long").alias("truck_id"),
        F.col("dc_id").cast("long").alias("dc_id"),
        F.col("store_id").cast("long").alias("store_id"),
        F.col("status"),
        F.col("event_ts"),
        F.col("eta"),
        F.col("etd"),
        F.col("departure_time"),
        F.col("actual_unload_duration").cast("decimal(9,2)").alias("actual_unload_duration"),
        F.col("trace_id"),
    )


def build_shipment_lines():
    return silver("fact_truck_inventory").select(
        F.col("shipment_id").alias("shipment_number"),
        F.col("truck_id").cast("long").alias("truck_id"),
        F.col("product_id"),
        F.col("quantity").cast("int").alias("quantity"),
        F.col("action"),
        F.col("location_id").cast("long").alias("location_id"),
        F.col("location_type"),
        F.col("event_ts"),
        F.col("trace_id"),
    )

In [ ]:
# =============================================================================
# LOAD PLAN  (parent -> child; FK-safe order)
# =============================================================================

LOAD_PLAN = [
    ("geographies",           build_geographies),
    ("products",              build_products),
    ("customers",             build_customers),
    ("stores",                build_stores),
    ("distribution_centers",  build_distribution_centers),
    ("trucks",                build_trucks),
    ("sales",                 build_sales),
    ("sale_lines",            build_sale_lines),
    ("online_orders",         build_online_orders),
    ("online_order_lines",    build_online_order_lines),
    ("payments",              build_payments),
    ("inventory_transactions", build_inventory_transactions),
    ("reorders",              build_reorders),
    ("stockouts",             build_stockouts),
    ("shipment_movements",    build_shipment_movements),
    ("shipment_lines",        build_shipment_lines),
]

if ONLY_TABLES:
    LOAD_PLAN = [(t, fn) for t, fn in LOAD_PLAN if t in ONLY_TABLES]

PLAN_TABLES = [t for t, _ in LOAD_PLAN]

In [ ]:
# =============================================================================
# WRITER  (silver -> blob CSV, RFC4180)
# =============================================================================
from pyspark.sql.types import BooleanType

CSV_OPTIONS = {
    "header": "true",
    "quote": '"',
    "escape": '"',
    "nullValue": "",
    "emptyValue": "",
    "encoding": "UTF-8",
    "timestampFormat": "yyyy-MM-dd HH:mm:ss.SSS",
    "dateFormat": "yyyy-MM-dd",
}


def _stamp_loaded_at(df):
    if "loaded_at" in df.columns:
        return df
    return df.withColumn("loaded_at", F.lit(LOADED_AT))


def _prep(df, table):
    """Stamp loaded_at, cast BIT-target booleans to 0/1, select columns in the
    exact DDL order the bulk-load SQL expects."""
    df = _stamp_loaded_at(df)
    for field in df.schema.fields:
        if isinstance(field.dataType, BooleanType):
            df = df.withColumn(field.name, F.col(field.name).cast("int"))
    return df.select(*[F.col(c) for c in TABLE_COLUMNS[table]])


def export_table(df, table):
    out = _prep(df, table)
    if SINGLE_FILE:
        out = out.coalesce(1)
    elif EXPORT_PARTITIONS:
        out = out.repartition(EXPORT_PARTITIONS)

    folder = f"{BLOB_BASE}/{table}"
    out.write.mode("overwrite").options(**CSV_OPTIONS).csv(folder)

    parts = [f.path for f in notebookutils.fs.ls(folder) if f.name.endswith(".csv")]
    if SINGLE_FILE:
        final = f"{BLOB_BASE}/{table}.csv"
        if notebookutils.fs.exists(final):
            notebookutils.fs.rm(final, True)
        notebookutils.fs.mv(parts[0], final, True)
        notebookutils.fs.rm(folder, True)
        return [f"{BLOB_PREFIX}/{table}.csv"]
    # multi-file: return blob-relative paths for the generated loader
    return [f"{BLOB_PREFIX}/{table}/{p.split('/')[-1]}" for p in parts]


def _with_clause(table):
    return ",\n        ".join(f"{c} {ty}" for c, ty in TABLE_DDL[table])


def generate_loader_sql(file_map):
    """Emit a chunked BULK INSERT loader per table. Azure SQL DB reads a single
    file per statement and does not support OPENROWSET+WITH for CSV over https, so
    each part file is bulk-loaded into a #stg_<table> temp table and immediately
    copied into the real table with INSERT...SELECT (then the staging table is
    truncated before the next part). Loading one part at a time keeps every
    transaction small -- a single INSERT of a 100M+ row staging table would bloat
    the log and can fail. The IDENTITY PK is server-assigned by the INSERT. Reuses
    the DATA_SOURCE/credential created by 10-bulk-load.sql's setup section."""
    opts = (
        "WITH (DATA_SOURCE = 'OltpBlobSrc', FORMAT = 'CSV', FIRSTROW = 2,\n"
        "      FIELDTERMINATOR = ',', FIELDQUOTE = '\"', ROWTERMINATOR = '0x0a', CODEPAGE = '65001');"
    )
    lines = [
        "-- Auto-generated loader (multi-file, chunked). Run 10-bulk-load.sql setup first.",
        "-- Replace nothing: paths are blob-relative to the external data source.",
        "",
    ]
    for table in PLAN_TABLES:
        paths = file_map.get(table, [])
        if not paths:
            continue
        cols = ", ".join(TABLE_COLUMNS[table])
        lines.append(f"IF OBJECT_ID('tempdb..#stg_{table}') IS NOT NULL DROP TABLE #stg_{table};")
        lines.append(f"CREATE TABLE #stg_{table} (\n        {_with_clause(table)}\n);")
        lines.append("GO")
        for path in paths:
            lines.append(f"BULK INSERT #stg_{table} FROM '{path}'")
            lines.append(opts)
            lines.append(f"INSERT INTO {{schema}}.{table} ({cols}) SELECT {cols} FROM #stg_{table};")
            lines.append(f"TRUNCATE TABLE #stg_{table};")
            lines.append("GO")
        lines.append(f"DROP TABLE #stg_{table};")
        lines.append("GO")
        lines.append("")
    return "\n".join(lines).replace("{schema}", "retail")


In [ ]:
# =============================================================================
# RUN
# =============================================================================
file_map = {}
for table, build in LOAD_PLAN:
    df = build()
    n = df.count()
    print(f"-> exporting {table}: {n:,} rows")
    file_map[table] = export_table(df, table)
    for p in file_map[table]:
        print(f"   wrote {p}")

print("\nExport complete.")
if not SINGLE_FILE:
    print("\n--- generated multi-file loader (save & run after 10-bulk-load.sql setup) ---\n")
    print(generate_loader_sql(file_map))
else:
    print("Run deploy/azuresql/bulk-load/10-bulk-load.sql to load these files.")
